# CSL module: `csl.py`

In [37]:
import os
import sys
module_path = os.path.abspath(os.path.join('../../..'))
if module_path not in sys.path:
    sys.path.append(module_path)
import gbtk.csl as csl
import pandas as pd
import numpy as np
import warnings

This module contains a description and functionality for a coincident site lattice (CSL). This is a common way of describing grain boundary systems and one that is especially well suited to simulations with periodic boundary conditions.

## Constructing a CSL
The following code instantiates a `csl` object with fcc lattice type:

In [38]:
test_csl = csl.CSL('fcc')

To obtain some useful feedback we will switch on the debug information for this csl (this information can get quite verbose so we can switch it off when using the library in scripts):

In [39]:
test_csl.set_debug()

We now use `h,k,l` to specify the misorientation axis and a pair of integers `m,n` to define the axis:

In [40]:
h = 1; k = 1; l = 1
m = 1; n = 5

In the cubic system, the integers `m` and `n` specify the misorientation angle via the formula

$$
\theta = 2 \tan^{-1}\left( \frac{m}{n} \sqrt (h^2+k^2+l^2)   \right).
$$



We can then use one of the helper functions to obtain the misorientation angle corresponding to `m,n` for axis `h,k,l`:

In [41]:
theta = csl.calculate_theta(h,k,l,m,n)
print(180*theta/np.pi)

38.213210701738184


The use of the integers `m` and `n` to specify a misorientation angle may seem obscure. If so, don't worry about it for now as further down we show how to catalogue possible CSLs for a given axis in a way that negates the need to understand the direct relevance of `m` and `n`.

Note that another helper function will give us the $\Sigma$ value for this boundary, using the formula

$$
\Sigma = n^2+m^2(h^2+k^2+l^2)
$$

with any factors of 2 eliminated to give an odd value for $\Sigma$:

In [42]:
print(csl.calculate_sigma(h,k,l,m,n))

7.0


Note that the above helper functions (and the equations) are only valid for cubic systems specified via cubic unit cells in `Lattice`.

We now use a couple of accessor functions to set the axis and angle of our csl:

In [43]:
test_csl.set_axis([h,k,l])
test_csl.set_angle(theta)

Actually, we can avoid the need to first calculate `theta` and instead set the misorientation angle directly with an alternative method:

In [44]:
test_csl.set_angle_mn(m,n)

Now we have specified our misorientation axis and angle, we want to do something useful. The code below does the following:

* Calculates the rotation matrix for our axis-angle combination
* Sets up the functionality required to search for a csl unit cell (using functionality in `spatialsearch.py`)
* Finds the csl basis

Because we have switched on the debug output we will get some useful feedback.

In [45]:
test_csl.find_misorientation_rotation_matrix()
test_csl.enable_search()
test_csl.find_csl_basis()

----------------------------------------------------
csl.find_csl_basis() debug:
***************************
CSL Cell vectors (lattice basis)
Black                      White
[      1      1      1 ]   [      1      1      1 ]
[      0      1     -2 ]   [      1      0     -2 ]
[     -1      2      0 ]   [      0      2     -1 ]
CSL Cell vectors (cartesian basis)
Black                                       White
[     1.000000     1.000000     1.000000]   [     1.000000     1.000000     1.000000 ]
[     0.000000     1.000000    -2.000000]   [     1.000000     0.000000    -2.000000 ]
[    -1.000000     2.000000     0.000000]   [     0.000000     2.000000    -1.000000 ]



True

The `find_csl_basis()` function returns details of the vectors of the CSL basis in both cartesian coordinates and the bassis of our specified lattice. These are equivalent here, since we are working with a cubic system.

This is a bit easier to understand if we visuallise the CSL basis.....

## Visualising the CSL 

At this stage we can use some of the visualisation tools to see what we have produced. First a look at the two csl unit cells in their original orientation:

In [46]:
test_csl.visualise_3d()

----------------------------------------------------
CSL visualisation:
******************
CSL cells contain     28 black atoms,     28 white atoms



This plot shows how we have carved out two regions of space from our original cubic lattice. These regions are exactly the same shape and share an edge parallel to the misorientation axis. These are the unit cells of our CSL.

Next we examine the two CSL unit cells rotated into one another. They should coincide exactly:

In [47]:
test_csl.visualise_3d_rotated()

----------------------------------------------------
CSL visualisation:
******************
CSL cells contain     28 black atoms,     28 white atoms



These CSL cells can now be tiled together to fill space. We could then define a boundary and on one side of it keep only the red atoms and on the other only the green atoms. But that is jumping ahead to what the `grainboundary` module does.

For now, we can take a look at a two-dimensional projection of the above data. The projection shown is determined by the optional argument `axis`. The default is `axis=-1` which will give a projection down the misorientation axis. Alternatively, values of 0, 1 or 2 will give projections down the corresponding csl unt cell vector.

In [48]:
test_csl.visualise_2d_rotated(axis=-1)

----------------------------------------------------
CSL visualisation:
******************
CSL cells contain     28 black atoms,     28 white atoms



This view should make things a bit clearer. You will see that our CSL unit cell contains two copies of the cubic lattice, but in different orientations. Some of the atoms of different colours coincide, but most don't. There are 28 atoms of each colour in the cell of which 4 coincide, i.e. 1/7th of the atoms coincide. You may recall from above that for this CSL $\Sigma = 7$. This is the meaning of $\Sigma$.

## Calculating the DSC lattice
In order to specify the microscopic degrees of freedom of a grain boundary, we need to know the details of the DSC lattice. `csl.py` can calculate this for us as below:

In [49]:
test_csl.find_dsc_lattice()

----------------------------------------------------
csl.find_dsc_lattice() debug:
*****************************
DSC Cell fractions (and reciprocals)
[     0.071429 (   14.000000)     0.071429 (   14.000000)     0.071429 (   14.000000) ]
DSC Cell vectors (cartesian basis)
Black                                       White
[     0.071429     0.071429     0.071429]   [     0.071429     0.071429     0.071429 ]
[     0.000000     0.071429    -0.142857]   [     0.071429     0.000000    -0.142857 ]
[    -0.071429     0.142857     0.000000]   [     0.000000     0.142857    -0.071429 ]



The debug output gives the DSC lattice as fractions of the CSL unit cell vectors. By definition the reciprocals of these fractions should be integers, as for the example above.

Once we have calculated the DSC lattice, the `visualise_2d_rotated()` function will automatically include it in the visualisation output (subject to a lower bound of 0.01 on the CSL cell fractions - otherwise the plot becomes too busy):

In [50]:
test_csl.visualise_2d_rotated(axis=-1)

----------------------------------------------------
CSL visualisation:
******************
CSL cells contain     28 black atoms,     28 white atoms



## Creating a CSL catalogue
An important part of the functionality in `csl.py` is the ability to catalogue possible axis-angle combinations. We will demonstrate this with an example. Assume we want to consider all possible boundaries with a $[1,1,1]$ misorientation axis with fewer than some particular number of atoms in the CSL unit cell. We do this with the following function:

`write_csl_catalogue(folder, lattice_type, axis, limit, ext='df', include_basis=False)`

The function will search a range of values of `m,n` to specify possible misorientation angles. The parameters are as follows (where not self-explanatory):
* `limit` is the upper bound to the values of `m,n`
* `ext` specifies the output file type, either the default pandas dataframe (`df`), or csv (`csv`), or excel (`xlsx`)
* `include_basis` is by default `False`. The code runs much faster in this mode. Setting the value to `True` will request that the CSL unit cells be found for each misorientation and the CSL cell vectors will be included in the file.

The following example writes a full catalogue (including CSL cell vectors) for a small range of `m,n`. The output is mostly self explanatory, but:

* `cos_num` and `cos_den` are the numerator and denominator of the cosine of the misorientation angle.
* `num_atoms` is the number of atoms in the CSL unit cell, and is only available if the CSL basis is calculated.
* `b_0[0]`, etc are the components of the two copies of the CSL unit cell (black and white copies) expressed in the original lattice basis`

In [56]:
h = 1; k = 0; l = 0
limit = 4
with warnings.catch_warnings(action="ignore"):
    csl.write_csl_catalogue('catalogues/csl/', 'fcc', [h,k,l], limit, include_basis=True)

There aren't many options here for values of `m` and `n` less than 4.

We can quickly look at the output by reloading the df file. Notice that the automatically chosen filename for the catalogue gives details of the specification:

In [52]:
pd.read_hdf('catalogues/csl/csl_cat_full_fcc_1_0_0_limit_4.df','df')

,h,k,l,m,n,theta,sigma,cos_num,cos_den,num_atoms,...,b_2[2],w_0[0],w_0[1],w_0[2],w_1[0],w_1[1],w_1[2],w_2[0],w_2[1],w_2[2]
0,1,0,0,1.0,4.0,28.072487,17.0,15.0,17.0,68.0,...,1.0,-1.0,0.0,0.0,0.0,1.0,4.0,0.0,4.0,-1.0
1,1,0,0,1.0,3.0,36.869898,5.0,4.0,5.0,20.0,...,-2.0,-1.0,0.0,0.0,0.0,-1.0,2.0,0.0,-2.0,-1.0
2,1,0,0,2.0,3.0,67.380135,13.0,5.0,13.0,52.0,...,-3.0,-1.0,0.0,0.0,0.0,-3.0,2.0,0.0,-2.0,-3.0
3,1,0,0,3.0,4.0,73.739795,25.0,7.0,25.0,100.0,...,4.0,-1.0,0.0,0.0,0.0,4.0,-3.0,0.0,3.0,4.0


We could consider writing a much larger catalogue. This is very fast if the CSL basis is not included:

In [53]:
with warnings.catch_warnings(action="ignore"):
    csl.write_csl_catalogue('catalogues/csl/', 'fcc', [1,0,0], limit=50)

And we can read it back in for a quick look:

In [54]:
pd.read_hdf('catalogues/csl/csl_cat_fcc_1_0_0_limit_50.df','df')

,h,k,l,m,n,theta,sigma,cos_num,cos_den
0,1,0,0,1.0,50.0,2.291526,2501.0,2499.0,2501.0
1,1,0,0,1.0,49.0,2.338279,1201.0,1200.0,1201.0
2,1,0,0,1.0,48.0,2.386979,2305.0,2303.0,2305.0
3,1,0,0,1.0,47.0,2.437750,1105.0,1104.0,1105.0
4,1,0,0,1.0,46.0,2.490729,2117.0,2115.0,2117.0
...,...,...,...,...,...,...,...,...,...
390,1,0,0,45.0,46.0,88.740803,4141.0,91.0,4141.0
391,1,0,0,46.0,47.0,88.767880,4325.0,93.0,4325.0
392,1,0,0,47.0,48.0,88.793818,4513.0,95.0,4513.0
393,1,0,0,48.0,49.0,88.818686,4705.0,97.0,4705.0


To proceed, you could now select combinations of axis and `m` and `n` to give, for example grain boundaries with misorentation angles of interest (perhaps low-angle boundaries) or with fewer atoms than a given limit (perhaps set by computational constraints).

The next step is to build some grain boundaries, using the `grainboundary` module.